# 03 · Agrupación de variables, WoE e IV

**Objetivo:** convertir cada variable en tramos de riesgo, medir qué tan predictiva es y elegir las que entrarán al scorecard.

Un **scorecard** es una tabla de puntos: cada característica del cliente (edad, uso de crédito, atrasos…) se divide en tramos y cada tramo da cierto número de puntos. La suma de puntos es la calificación del cliente, y a cada calificación le corresponde una **PD**. Este notebook construye los tramos; el notebook 04 les asigna los puntos.

## Conceptos

### 1. Agrupación de variables (*binning*)

Se hace en dos pasos:
- **Tramos finos (*fine classing*):** se corta cada variable en muchos tramos pequeños (hasta 20, por cuantiles).
- **Tramos gruesos (*coarse classing*):** se fusionan tramos vecinos hasta cumplir cuatro reglas:
  1. **Tamaño mínimo:** cada tramo tiene al menos 5 % de los clientes. Un tramo con pocos clientes da estimaciones inestables.
  2. **Tendencia monótona:** la tasa de malos siempre sube o siempre baja. Así el scorecard tiene lógica de negocio ("a mayor uso de crédito, más riesgo") y no premia ruido. La excepción son variables con una relación en U justificada por el negocio.
  3. **Máximo de 8 tramos:** un scorecard simple es más fácil de explicar y más estable en el tiempo.
  4. **Faltantes en su propio tramo**, como se decidió en la etapa 2.

### 2. WoE (*Weight of Evidence*, peso de la evidencia)

Para cada tramo:

$$\text{WoE} = \ln\left(\frac{\%\ \text{de los buenos que caen en el tramo}}{\%\ \text{de los malos que caen en el tramo}}\right)$$

| WoE | Significado |
|---|---|
| Positivo | El tramo concentra más buenos que malos: **menos riesgo** que el promedio. |
| Cero | Riesgo igual al promedio. |
| Negativo | El tramo concentra más malos que buenos: **más riesgo** que el promedio. |

**¿Por qué transformar a WoE?**
- **Captura relaciones no lineales:** en la etapa 2 vimos formas en U y saltos en un extremo; con WoE cada tramo recibe un valor según su riesgo real.
- **Maneja faltantes y extremos de forma natural:** cada uno es solo otro tramo.
- **Pone todas las variables en la misma escala** (logaritmo de momios, la misma que usa la regresión logística), lo que facilita convertir el modelo en puntos.

### 3. IV (*Information Value*, valor de la información)

Mide el poder predictivo de toda la variable:

$$\text{IV} = \sum_{\text{tramos}} (\%\ \text{buenos} - \%\ \text{malos}) \times \text{WoE}$$

| IV | Poder predictivo |
|---|---|
| Menos de 0.02 | Sin poder predictivo |
| 0.02 a 0.1 | Débil |
| 0.1 a 0.3 | Medio |
| 0.3 a 0.5 | Fuerte |
| Más de 0.5 | Muy fuerte: se revisa que no haya fuga de información |

## Entorno

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # Donde está cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config
from src.datos import copiar_datos_crudos
from src.eda import graficar_tasa_malos
from src.limpieza import obtener_datos_limpios
from src.woe import (
    NO_MONOTONICAS,
    VARIABLES_CANDIDATAS,
    AgrupadorWoE,
    graficar_iv,
    preparar_woe,
    seleccionar_variables,
)

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)

In [ ]:
# Datos limpios de la etapa 2 (se regeneran si es una sesión nueva de Colab)
copiar_datos_crudos(CARPETA_DRIVE)
train, test = obtener_datos_limpios()

Y = config.COLUMNA_OBJETIVO
TASA_GLOBAL = train[Y].mean()
print(f"Entrenamiento: {len(train):,} clientes | Prueba: {len(test):,} clientes")

## 1. Agrupación de todas las variables

El agrupador aprende los tramos **solo con entrenamiento**, como el limpiador de la etapa 2. `num_creditos_inmobiliarios` no se fuerza a tendencia monótona por su forma en U: tener 0 créditos inmobiliarios (sin patrimonio) o 4 o más (sobreendeudamiento) es más riesgoso que tener 1 o 2.

In [ ]:
agrupador = AgrupadorWoE(
    n_tramos_iniciales=20,     # Tramos finos
    pct_minimo=0.05,           # Mínimo 5 % de clientes por tramo
    max_tramos=8,              # Máximo de tramos por variable
    min_clientes_faltante=50,  # Faltantes con menos clientes reciben WoE = 0
    no_monotonicas=NO_MONOTONICAS,
).fit(train, VARIABLES_CANDIDATAS)

print("Variables sin tendencia monótona forzada:", NO_MONOTONICAS)

## 2. Ejemplo paso a paso: edad

Antes de ver todas las variables, se revisa una tabla completa para entender cada columna:
- `clientes`, `malos`, `buenos`: conteos del tramo.
- `pct_clientes`: proporción de clientes en el tramo.
- `tasa_malos`: malos ÷ clientes del tramo.
- `dist_buenos`: qué parte de **todos los buenos** cae en el tramo.
- `dist_malos`: qué parte de **todos los malos** cae en el tramo.
- `woe` = ln(dist_buenos ÷ dist_malos).
- `iv` = (dist_buenos − dist_malos) × woe. La suma de esta columna es el IV de la variable.

In [ ]:
tabla_edad = agrupador.tablas_["edad"]
tabla_edad

In [ ]:
# El WoE del primer tramo, calculado a mano para comprobar la fórmula
fila = tabla_edad.iloc[0]
dist_buenos = fila["buenos"] / tabla_edad["buenos"].sum()
dist_malos = fila["malos"] / tabla_edad["malos"].sum()
woe = np.log(dist_buenos / dist_malos)

print(f"Tramo: {tabla_edad.index[0]}")
print(f"  % de todos los buenos en el tramo: {dist_buenos:.4f}")
print(f"  % de todos los malos en el tramo:  {dist_malos:.4f}")
print(f"  WoE = ln({dist_buenos:.4f} / {dist_malos:.4f}) = {woe:.4f}")
print(f"\nIV de edad (suma de la columna iv): {tabla_edad['iv'].sum():.4f}")

## 3. IV de todas las variables

In [ ]:
resumen_iv = agrupador.resumen_iv()
resumen_iv

In [ ]:
graficar_iv(resumen_iv)
plt.show()

## 4. Tramos finales por variable

Para cada variable: la tabla de tramos con su WoE y la tasa de malos por tramo. Lo que se revisa:
- Que la tendencia tenga sentido de negocio.
- Que ningún tramo sea demasiado pequeño.
- Cómo se comporta el tramo "Faltante".

In [ ]:
columnas = ["clientes", "pct_clientes", "malos", "tasa_malos", "woe", "iv"]

for variable in resumen_iv.index:
    tabla = agrupador.tablas_[variable]
    print(f"\n{variable}  |  IV = {tabla['iv'].sum():.4f}")
    display(tabla[columnas])
    graficar_tasa_malos(tabla, f"Tasa de malos por tramo final de {variable}", TASA_GLOBAL)
    plt.show()

## 5. Selección de variables

Dos filtros, en orden:
1. **IV mínimo de 0.02:** las variables sin poder predictivo solo agregan ruido.
2. **Redundancia:** si dos variables transformadas a WoE tienen correlación mayor a 0.7, dicen casi lo mismo y se conserva la de mayor IV. Esto evita la **multicolinealidad** en la regresión logística.

In [ ]:
train_woe = agrupador.transform(train)
test_woe = agrupador.transform(test)   # Mismos tramos y WoE aprendidos en entrenamiento

seleccion, decisiones = seleccionar_variables(resumen_iv, train_woe)
decisiones

In [ ]:
# Correlación entre las variables candidatas ya transformadas a WoE
candidatas = resumen_iv[resumen_iv["iv"] >= 0.02].index
train_woe[candidatas].corr().round(2)

In [ ]:
print(f"Variables seleccionadas ({len(seleccion)}):")
for variable in seleccion:
    print(f"  - {variable}  (IV = {resumen_iv.loc[variable, 'iv']:.4f})")

## 6. Guardar datos transformados

Se guardan entrenamiento y prueba con las variables seleccionadas en WoE. La función `preparar_woe()` de `src/woe.py` repite todo este proceso con los mismos parámetros, así el notebook 04 puede reconstruir los tramos en una sesión nueva. Aquí se comprueba que da exactamente el mismo resultado.

In [ ]:
columnas_modelo = seleccion + [Y]
train_woe[columnas_modelo].to_csv(config.ARCHIVO_TRAIN_WOE)
test_woe[columnas_modelo].to_csv(config.ARCHIVO_TEST_WOE)

# Comprobación de reproducibilidad
_, seleccion_repetida, train_repetido, _ = preparar_woe()
assert seleccion_repetida == seleccion
assert np.allclose(train_repetido[seleccion], train_woe[seleccion])
print("Reproducible: preparar_woe() da los mismos tramos, WoE y variables.")
print("Guardado:", config.ARCHIVO_TRAIN_WOE.relative_to(config.RAIZ), "y",
      config.ARCHIVO_TEST_WOE.relative_to(config.RAIZ))

## Resumen de la etapa

- Cada variable se agrupó en tramos con reglas de tamaño mínimo, tendencia monótona, máximo de tramos y faltantes como tramo propio, **aprendidas solo con entrenamiento**.
- Cada tramo recibió su WoE y cada variable su IV.
- Se eligieron las variables con IV de al menos 0.02 y sin redundancia.
- Se guardaron `train_woe.csv` y `test_woe.csv` en `data/processed/`.

**Siguiente paso:** notebook `04_scorecard.ipynb`, regresión logística sobre las variables WoE y conversión a puntos.